# TP03 — Track, Compare and Recover ML Experiments with MLflow

**ESILV — Machine Learning Operations**
**Duration:** 90 minutes · **Dataset:** IBM Telco Customer Churn · **Starting point:** TP02 project and its scikit-learn Pipeline

> Your team has trained several churn models. A colleague asks: "Which model did you choose? What produced it? Can I load it without running training again?" Provide evidence.

This notebook keeps the subject of the TP. After each section, a ✅ block says where it was done in this repository, with the command to re-run it.

| Deliverable | File |
|---|---|
| Summary with screenshots | `reports/tp03_summary.md` |
| Answers to the questions | `evidence/tp03/answers.md` |
| Identifiers (run, model URI, version, alias) | `evidence/tp03/handover.json` |

In [ ]:
# Commands below run from the project root (two levels above this notebook).
import os
from pathlib import Path

if not Path("pyproject.toml").exists():
    os.chdir(Path.cwd().parents[1])
print(Path.cwd())

## Mission

Extend the existing training workflow with MLflow. By the end, it should be possible to find a training run, inspect its configuration and validation results, recover its complete fitted pipeline, and hand it over through a registered model alias.

Keep the scope manageable: reuse the TP02 models, preprocessing and evaluation functions. No new algorithms or large hyperparameter searches are needed.

### Learning outcomes

- Explain Experiment, Run, Parameter, Metric, Artifact and Model.
- Add explicit tracking to an existing training program.
- Compare runs under the same evaluation protocol.
- Connect a model to its code and data identities.
- Distinguish database metadata, artifact files and dataset records.
- Register the selected model, assign an alias and load it in a fresh process without refitting.

### Before starting

- The TP02 project must already train at least one complete preprocessing-and-classifier pipeline from a terminal.
- Same six features: `tenure`, `MonthlyCharges`, `TotalCharges`, `Contract`, `InternetService`, `PaymentMethod`. `Churn = Yes` → class `1`; the target and `customerID` are not features.
- Keep the TP02 train/validation/test allocation. Fit preprocessing on training only.
- Compare candidates on validation with **average precision as the primary metric** (continuous positive-class scores). Also report precision, recall and F1 at the same documented threshold. Keep the test set reserved.
- Do not switch metrics after seeing which model wins.

### Suggested timing

| Time | Milestone |
|---|---|
| 0–15 min | Start the server, create a run and inspect storage |
| 15–40 min | Record parameters, metrics, artifacts and dataset inputs |
| 40–55 min | Compare at least three existing candidate configurations |
| 55–80 min | Log, register and reload the selected pipeline through an alias |
| 80–90 min | Restart, verify persistence and collect handover evidence |

## 1. Start tracking locally — 15 minutes

Add MLflow to the environment, from the project root:

```bash
uv add "mlflow>=3,<4"
uv run mlflow --version
```

Use MLflow 3.x and keep the resolved version in the lockfile.

In a first terminal, from the project root (keep it running, UI at http://127.0.0.1:5000):

```bash
uv run mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db --artifacts-destination ./mlartifacts
```

The server provides the tracking API and the UI. SQLite stores run metadata; the artifact directory stores files such as recorded models.

| Component | What it holds |
|---|---|
| Backend store: `mlflow.db` | Experiment/run records, parameters, metrics, tags and registry metadata |
| Artifact store: `mlartifacts/` | Model files, confusion matrices and other logged files |
| Tracking server | Receives logging/query requests and serves the UI |
| Training client | Executes the training code and sends records to the server |

Do not modify database rows or generated artifact files manually.

**Question:** if a colleague receives only `mlflow.db`, what else would they need to load the recorded pipeline?

Code hint — connect the training client:

```python
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("telco-churn-tp03")
```

Create a small first run with `mlflow.start_run()` and log one parameter. Find it in the UI.

**Checkpoint:** point to the experiment and the ID of one run.

**Explain:** what is the difference between an experiment and a run? Is the browser page executing the training code?

> ✅ **In this repo:** MLflow 3.18.0 in `pyproject.toml` / `uv.lock`. Connection in `src/churnlab/tracking.py` (`connect`). First run: `scripts/tracking_smoke.py`.

In [ ]:
# Requires the tracking server to be running in another terminal.
!uv run python scripts/tracking_smoke.py

## 2. Record what produced each result — 25 minutes

Wrap one candidate's training and validation evaluation in one run (one run per candidate). Use explicit logging.

| Category | Required information |
|---|---|
| Parameters | Estimator type, important hyperparameters, split seed, decision threshold |
| Metrics | `val_average_precision`, `val_precision`, `val_recall`, `val_f1` |
| Tags | Git commit and SHA-256 of the original CSV |
| Artifacts | Validation confusion matrix and a compact configuration/provenance JSON |
| Dataset inputs | Training and validation dataset metadata with distinct contexts |
| Selected model (section 4) | Complete fitted preprocessing-and-classifier pipeline |

The JSON should also identify feature names, class mapping, split/customer-allocation reference and evaluation rules. A file hash identifies bytes; it does not store or recover the CSV.

Code hint:

```python
with mlflow.start_run(run_name=run_name) as run:
    mlflow.log_params({"model_type": ..., "random_state": ..., "threshold": ...})
    mlflow.set_tags({"git_commit": ..., "data_sha256": ...})
    # fit the complete pipeline on training data
    # evaluate validation data with the TP02 evaluation function
    mlflow.log_metrics(validation_metrics)
    mlflow.log_artifact(str(confusion_matrix_path))
    mlflow.log_dict(provenance, "provenance.json")
    print("Run ID:", run.info.run_id)
```

- Pass only actual candidate settings.
- Find the class-1 score column from `classes_`. AP uses continuous scores; precision, recall and F1 use the fixed threshold. State the confusion matrix class ordering.

Associate data with the run (do not change the split):

```python
train_dataset = mlflow.data.from_pandas(train_df, name="telco_train", targets="Churn")
mlflow.log_input(train_dataset, context="training")
# + a validation dataset with context="validation"
```

Dataset records describe the data used; they do **not** store the DataFrame or version it. The MLflow digest is not the original-file SHA-256.

**Question:** what evidence distinguishes two different train/validation allocations made from the same original CSV?

**Checkpoint:** in one run, locate parameters, metrics, tags, files and training/validation dataset records. Explain the difference between a metric and an artifact.

**Think about it**
- Does recording a random seed guarantee identical results on every machine?
- Can someone recover the training data from its hash alone?
- What happens to traceability if a commit is logged while training uncommitted edits?

Commit the training changes before the final comparison runs.

> ✅ **In this repo:** `track_candidate` in `src/churnlab/tracking.py` (params, metrics, tags, confusion matrix, `provenance.json`, datasets, model). Loop over the TP02 candidates in `scripts/train_tracked.py`. Code committed in d51e0ff before the runs.

## 3. Compare at least three candidate runs — 15 minutes

Produce at least three completed candidate runs (different TP02 classifiers or configurations). Keep the six features, customer allocation, evaluation definitions and threshold policy fixed. Use descriptive run names.

Select the runs in the UI, compare them, and fill a decision table:

| Run ID | Candidate / changed setting | Validation AP | Validation recall | Validation F1 | Decision |
|---|---|---:|---:|---:|---|

Choose using the predefined primary metric. One split is a classroom comparison, not proof of a significant improvement.

**Checkpoint:** screenshot of the comparison + 3–5 sentences explaining the selection.

**Explain:** why would results from different validation populations be harder to compare? Why avoid selecting repeatedly on the test set?

> ✅ **In this repo:** 4 runs (`baseline-majority`, `logistic-C1`, `tree-depth5-leaf20`, `forest-300trees-depth8`). Table in `evidence/tp03/comparison.csv`, screenshot in `evidence/tp03/screenshots/compare_runs.png`. Selected: random forest (AP 0.664).

In [ ]:
# Runs one tracked training: creates 4 new runs and a NEW registered version.
# Only needed to reproduce the experiment; the evidence already exists.
# !uv run python scripts/train_tracked.py

# Read-only: lists the runs, versions and alias stored on the server.
!uv run python scripts/tracking_status.py

## 4. Log, register and recover the complete model — 25 minutes

Record the fitted preprocessing + classifier together with `mlflow.sklearn.log_model` (a `joblib` file logged as a generic artifact does not count). At least the selected pipeline must be logged in its own run.

```python
model_info = mlflow.sklearn.log_model(
    sk_model=fitted_pipeline,
    name="churn_pipeline",
    input_example=X_train.head(3),
    serialization_format="cloudpickle",
)
print("Model URI:", model_info.model_uri)
```

Use original, unencoded feature columns for the example (no `Churn`, no `customerID`). Inspect the stored input schema. Record **both the run ID and the model URI**.

### Register for handover

Register the selected pipeline as `telco-churn` and give the selected version the alias `candidate` (UI is enough; API optional):

```python
version = mlflow.register_model(model_info.model_uri, "telco-churn")
client = mlflow.MlflowClient()
# client.set_registered_model_alias(...) -> "candidate" on version.version
```

Record the actual returned version.

| Identifier | Question it answers |
|---|---|
| Run ID | Which execution produced the result? |
| Logged model URI | Which recorded model artifact should be loaded? |
| Registered name + version | Which version belongs to `telco-churn`? |
| Alias `candidate` | Which version is currently selected for this role? |

An alias is a mutable reference, not a copy and not a deployment.

### Load through the alias in a fresh process

```python
mlflow.set_tracking_uri("http://127.0.0.1:5000")
loaded_pipeline = mlflow.sklearn.load_model("models:/telco-churn@candidate")
predictions = loaded_pipeline.predict(new_customers)
```

Run it in a **new terminal process**, without `fit` and without importing training code. Apply the same input parsing as TP02. Compare original and reloaded predictions on the same records.

**Checkpoint:** source run ID, model URI, registered version, alias, inference command and prediction agreement. Why is loading in a new process stronger evidence than predicting again in the training notebook?

**Question:** if the alias later points to a new version, does an already-loaded Python object change?

> ✅ **In this repo:** Logged in every run (`churn_pipeline`). Registered via the API in `scripts/train_tracked.py`: `telco-churn` v1, alias `candidate`. Identifiers in `evidence/tp03/handover.json`. Inference: `scripts/predict_registered.py`.

In [ ]:
# Inference only, in a separate process: loads models:/telco-churn@candidate, no fit.
!uv run python scripts/predict_registered.py --input evidence/tp03/inference_batch.csv --reference evidence/tp03/reference_predictions.csv

In [ ]:
import json

json.loads(Path("evidence/tp03/handover.json").read_text())

## 5. Verify persistence, submit and explain — 10 minutes

Stop the tracking server and restart it from the same project root with the same command. Check that runs, the registered version and the alias are still there, then run the inference again.

Do not delete storage to simulate failures. Answer instead: what becomes unavailable if the database survives but the model files are lost? What is missing if only the model files survive?

**Checkpoint:** short before/after restart observation.

Update the README with the commands (server, tracked training, inference), where tracking data are stored and how to recover the model.

`.gitignore`:

```gitignore
.venv/
mlflow.db
mlflow.db-*
mlartifacts/
mlruns/
```

Keep the local database and artifacts for the demo (ignored by Git ≠ deleted). A run ID or local model URI alone is not enough for someone on another machine: document this limitation.

### Deliverables — `reports/tp03_summary.md`

1. Screenshot comparing at least three runs + selection table.
2. Selected-run screenshot: parameters, validation metrics, training/validation datasets.
3. Registry screenshot: `telco-churn`, selected version, `candidate` alias.
4. Source run ID, model URI, registered version, alias, Git commit, data hash.
5. Selection reason, fresh-process inference command, prediction agreement.
6. Restart observation + what the database and artifact store each preserve, and what a colleague on another machine must obtain separately.

> ✅ **In this repo:** Restart checked: `evidence/tp03/status_before_restart.txt` and `status_after_restart.txt` are identical. README section "TP03". `.gitignore` updated. Summary: `reports/tp03_summary.md`.

## Assessment

| Criterion | Points |
|---|---:|
| Local setup, Experiment / Run organization and storage/restart explanation | 1 |
| Useful parameters, metrics, tags, artifacts and dataset inputs | 3 |
| Three fair candidate runs and justified validation-based choice | 2 |
| Complete logged pipeline, registered version/alias, fresh-process reload and agreement | 3 |
| Clear README, provenance and explanation of limitations | 1 |
| **Total** | **10** |

## If stuck

| Symptom | First check |
|---|---|
| Connection refused | Is the server running? Same port in the client? |
| Run missing in the UI | Tracking URI and experiment in the training process |
| Port 5000 occupied | Use 5001 everywhere |
| Runs disappear after restart | Restarted from the same root with the same SQLite path? |
| Loaded model rejects inputs | Compare columns/types with the recorded signature |
| Alias cannot be resolved | Registry name, alias spelling, version, tracking URI |
| Dataset input missing | `log_input` inside the active run? |

## Optional extensions

- Registry automation through the API (done here), second version and alias move.
- `mlflow.search_runs` export (done here: `evidence/tp03/search_runs.csv`).
- `mlflow.sklearn.autolog` in a separate experiment, compared with explicit logging.

## Documentation

- [MLflow CLI](https://mlflow.org/docs/latest/api_reference/cli.html)
- [Tracking server and artifact storage](https://mlflow.org/docs/latest/self-hosting/architecture/tracking-server/)
- [MLflow scikit-learn API](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.sklearn.html)
- [Model signatures and input examples](https://mlflow.org/docs/latest/ml/model/)
- [Model Registry](https://mlflow.org/docs/latest/ml/model-registry/workflow)
- [Dataset tracking API](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.data.html)